# Using a Custom Model

This notebook shows how to plug a model of your own into bacpipe's pipeline. You
define a model class that subclasses `ModelBaseClass`, and bacpipe takes care of
everything else: loading and windowing the audio, batching, saving the embeddings
in the standard folder structure, and training/evaluating probes on top of them.

Make sure to reset your config and settings files before running this notebook, 
as they may contain settings from previous runs that could prevent this notebook 
from finding the correct paths.

---
## 1. Setup & Configuration
Import necessary modules 

In [10]:
# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython

from IPython.display import display
import os 
from pathlib import Path

# load the specific package
import bacpipe

Set the working directory to the repository root and clean the previous tests if needed/wanted

In [11]:
import importlib.resources as pkg_resources
os.chdir(pkg_resources.files("bacpipe"))
os.chdir('..')
print(os.listdir('.'))


# Change the value of the key main_results_dir in the namespace bacpipe.settings to change the directory 
# where the results of the tutorials are stored. By default, it is set to './bacpipe_results'.
bacpipe.settings.main_results_dir = str(Path(bacpipe.settings.main_results_dir) / 'using_a_custom_model')

# !WARNING! the following code deletes the folder where the results of this tutorial is stored to be sure to start with a clean folder. 
# If you have important data in this folder, please comment it before running this code.
folder_path = bacpipe.settings.main_results_dir
if os.path.exists(folder_path):
    # Prompt the user
    user_input = input(f"Are you sure you want to delete '{folder_path}'? (y/n): ").lower().strip()

    if user_input == 'y':
        import shutil
        shutil.rmtree(folder_path) 
        print(f"Folder {folder_path} deleted.")
    else:
        print("Operation cancelled.")

else:
    print(f"Folder {folder_path} not found.")

['.vscode', 'bacpipe_results', 'run_pipeline.py', 'embed_obj = bacpipe.py', '.gitignore', 'bacpipe', '.readthedocs.yml', '.python-version', '.pytest_cache', 'pyproject.toml', 'bacpipe_model_checkpoints', '.github', '.coverage', '.venv', 'LICENSE', 'requirements_no_cuda.txt', 'requirements_tf_gpu.txt', '.mypy_cache', 'env_build', '.gitattributes', 'requirements_no_tf.txt', '.pre-commit-config.yaml', 'src', 'dist', '.git', 'README.md', 'docs', 'uv.lock']
Folder bacpipe_results/using_a_custom_model/using_a_custom_model not found.


Set the global constants used throughout the notebook.

In [12]:
MODEL_NAME = 'mel'                      # Choose the name of the custom model to run.
AUDIO_DIR = 'bacpipe/tests/test_data'   # path to directory containing audio files

---
## 2. Create a new model from scratch

Define your own model by subclassing `ModelBaseClass` and plug it directly into
bacpipe's pipeline.

The contract with bacpipe is:

- `SAMPLE_RATE`: the sample rate (in Hz) of the audio the model expects.
- `SEGMENT_LENGTH`: the input length (in samples) of one processing window, e.g.
  `48000 * 3` for 3 seconds at 48 kHz. Bacpipe windows the audio into segments of
  this length and pads the final segment if needed.
- `preprocess(self, audio)`: receives a `torch.Tensor` window and returns it in
  the format the model expects.
- `__call__(self, audio)`: receives the preprocessed window and must return a 2D
  array of features/embeddings with shape `(number_of_windows, feature_dimension)`.

Here the custom model computes a mel-spectrogram with `librosa`. Because no
deep-learning weights are involved, the "embeddings" are the raw spectrogram
features. Each window's mel-spectrogram is flattened to a 1D vector, as bacpipe
expects one embedding vector per window.


In [13]:
import librosa as lb
from bacpipe.model_pipelines.model_utils import ModelBaseClass

class MyModel(ModelBaseClass):
    SAMPLE_RATE = 48000         # the sample rate of the audio files that will be processed by the model
    SEGMENT_LENGTH = 48000*3    # 3s => the length of the audio segments that will be processed by the model (in samples)

    def __init__(self, **kwargs):
        super().__init__(sr=self.SAMPLE_RATE, segment_length=self.SEGMENT_LENGTH, **kwargs)

    def preprocess(self, audio):
        return audio

    def __call__(self, audio):
        audio = audio.cpu().numpy()
        mel_spec = lb.feature.melspectrogram(y=audio, sr=self.SAMPLE_RATE)
        # flatten each window's mel-spectrogram into a single 1D feature vector
        return mel_spec.reshape(len(mel_spec), -1)

---
## 3. Using this new built-in model

1. First generate the feature vectors (which are mels in this case, and not the
   embeddings of a deep-learning model).
2. Train a probe to evaluate the quality of the feature vectors (or embeddings)
   of the model.

`run_pipeline_for_single_model` accepts the custom class through the
`CustomModel` keyword argument. The name given in `model_name` is the key under
which the results are stored. The rest of the pipeline is identical to the
built-in models: `ground_truth_by_model` connects the embeddings to the
annotations, and `probing_pipeline` trains and evaluates a linear probe on the
custom features. `metrics['overall']` then quantifies how well your custom
features separate the annotated species.


In [14]:
# load the data to be process as well the model and compute the embeddings
loader_obj = bacpipe.run_pipeline_for_single_model(
    model_name=MODEL_NAME,              # name of the model to run. 
    audio_dir=AUDIO_DIR,                # path to directory containing audio files  
    CustomModel=MyModel
)

# get the computed embeddings as an array
embeds = loader_obj.embeddings(return_type='array')

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

INFO:bacpipe:Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.




###### Generating embeddings using MEL ######

INFO:bacpipe:


###### Generating embeddings using MEL ######

finding audio files: 11it [00:00, 24502.04it/s]
Found 7 number of audio files.
INFO:bacpipe:Found 7 number of audio files.
Using device='cpu'
INFO:bacpipe:Using device='cpu'
Skipping model.eval() because model is from tensorflow.
ERROR:bacpipe:Skipping model.eval() because model is from tensorflow.
                                                                  

In [15]:
# Run this function after computing the embeddings otherwise it is no able to find the connection between embeddings and labels
gt = bacpipe.ground_truth_by_model(
    model=MODEL_NAME, 
    audio_dir=AUDIO_DIR,
    annotations_filename='annotations.csv',
    overwrite=False
)

# Train and test a new probe associated to the feature vectors of the model 
# and evaluate the performance of the probe. 
# The returned metrics are the same as for the evaluation of a classification model, 
# but here they are used to evaluate the quality of the embeddings of the model.
probe, label2idx, metrics = bacpipe.probing_pipeline(
    model_name=MODEL_NAME, 
    ground_truth=gt,
    embeds=embeds
    )

# display the main metrics of the probe
display(metrics['overall'])

                                                                                                  
The simultaneous labels column of the ground truth has values exceeding 1. This means you have multi-label ground truth annotations. If this should not be happening ensure the ground truth is created correcly.

The simultaneous labels column of the ground truth has values exceeding 1. This means you have multi-label ground truth annotations. If this should not be happening ensure the ground truth is created correcly.

                                                                                                  
The simultaneous labels column of the ground truth has values exceeding 1. This means you have multi-label ground truth annotations. If this should not be happening ensure the ground truth is created correcly.

The simultaneous labels column of the ground truth has values exceeding 1. This means you have multi-label ground truth annotations. If this should not be happening ensu

{'macro_accuracy': 0.5, 'auc': 0.925, 'macro_f1': 0.41666666666666663}

---
## 4. Full Pipeline - Multiple Models

`run_pipeline_for_models` runs the full pipeline across several models in a single
call — the multi-model counterpart of `run_pipeline_for_single_model`. Custom
models are passed as a list through the `CustomModels` keyword, with one entry
per model name and in the same order as the `models` list: built-in models get a
`None` entry, custom models get their class.

Here the mel-spectrogram custom model (`MyModel`) is combined with the built-in
`perch_bird` model. Both are stored in the same results folder, so the two
models can be compared on exactly the same audio windows.



In [16]:
# combine the custom model with a built-in one: CustomModels is parallel to models
loader_dictionary = bacpipe.run_pipeline_for_models(
    models=[MODEL_NAME, 'perch_bird'],                  # custom model + built-in model
    audio_dir=AUDIO_DIR,                                # path to directory containing audio files
    dim_reduction_model='umap',                         # dimensionality reduction model to use for visualization
    CustomModels=[MyModel, None],                       # one class per model; None for built-in models
)

# the returned dictionary is keyed by model name and holds a Loader per model
display(loader_dictionary[MODEL_NAME].metadata_dict)

# each Loader exposes the same methods as before
print('mel embeddings shape:       ',
      loader_dictionary[MODEL_NAME].embeddings(return_type='array').shape)
print('perch_bird embeddings shape:',
      loader_dictionary['perch_bird'].embeddings(return_type='array').shape)

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

INFO:bacpipe:Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.




###### Generating embeddings using MEL ######

INFO:bacpipe:


###### Generating embeddings using MEL ######

finding audio files: 11it [00:00, 32814.61it/s]
Found 7 number of audio files.
INFO:bacpipe:Found 7 number of audio files.
Using device='cpu'
INFO:bacpipe:Using device='cpu'
Skipping model.eval() because model is from tensorflow.
ERROR:bacpipe:Skipping model.eval() because model is from tensorflow.
                                                                  Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

INFO:bacpipe:Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.




###### Generating embeddings using UMAP ######

INFO:bacpipe:


###### Generating embeddings

{'model_name': 'mel',
 'audio_dir': 'bacpipe/tests/test_data',
 'embed_dir': 'bacpipe_results/using_a_custom_model/using_a_custom_model/test_data/embeddings/2026-08-26_16-47___mel-test_data',
 'files': {'audio_files': ['audio/FewShot/CHE_01_20190101_163410.wav',
   'audio/FewShot/CHE_02_20190101_183410.wav',
   'audio/FewShot/CHE_03_20190201_163410.wav',
   'audio/FewShot/CHE_04_20190203_175410.wav',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031300.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031400.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031500.WAV'],
  'file_lengths (s)': [63.98977083333333,
   9.890729166666667,
   8.202479166666667,
   9.351708333333333,
   30.0,
   30.0,
   30.0],
  'nr_embeds_per_file': [22, 4, 3, 4, 10, 10, 10]},
 'segment_length (samples)': 144000,
 'sample_rate (Hz)': 48000,
 'embedding_size': 36096,
 'nr_embeds_total': 63,
 'total_dataset_length (s)': 181.4346875}

mel embeddings shape:        (63, 36096)
perch_bird embeddings shape: (37, 1280)


---
## 5. End-to-End: `bacpipe.play`

`bacpipe.play` is the highest-level entry point. It runs the complete pipeline —
embeddings, classification, optional dimensionality reduction, evaluation and an
interactive dashboard — in a single call. Custom models are passed exactly as in
`run_pipeline_for_models`, through the `CustomModels` keyword.

The dashboard is disabled here (`dashboard=False`) so that the notebook runs
headlessly; set it to `True` to launch the interactive dashboard at
http://localhost:5006. Because the embeddings were already computed in the
previous section, this rerun detects them and only recomputes what is missing.

In [17]:
bacpipe.play(
    models=[MODEL_NAME, 'perch_bird'],   # custom model + built-in model
    audio_dir=AUDIO_DIR,                 # path to directory containing audio files
    dim_reduction_model='umap',          # dimensionality reduction model to use for visualization
    CustomModels=[MyModel, None],        # one class per model; None for built-in models
    dashboard=False,                     # set to True to launch the interactive dashboard
)

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

INFO:bacpipe:Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.



---
## 6. High-Level Workflow - `generate_embeddings`

`bacpipe.generate_embeddings` is the single-model building block behind
`run_pipeline_for_single_model` and `run_pipeline_for_models`. It accepts the
custom class through the `CustomModel` keyword as well. Because the embeddings
were already computed above, this call simply loads them from disk.

In [18]:
loader_obj = bacpipe.generate_embeddings(
    model_name=MODEL_NAME,
    audio_dir=AUDIO_DIR,
    CustomModel=MyModel,
)

display(loader_obj.metadata_dict)

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

INFO:bacpipe:Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.




###### Generating embeddings using MEL ######

INFO:bacpipe:


###### Generating embeddings using MEL ######

Finding all generated embeddings: 7it [00:00, 20560.31it/s]
Found 7 embedding files.
INFO:bacpipe:Found 7 embedding files.
finding audio files: 11it [00:00, 31173.88it/s]
Found 7 number of audio files.
INFO:bacpipe:Found 7 number of audio files.

### Embeddings already exist. Using embeddings in bacpipe_results/using_a_custom_model/using_a_custom_model/test_data/embeddings/2026-08-26_16-47___mel-test_data ###
INFO:bacpipe:
### Embeddings already exist. Using embeddings in bacpipe_results/using_a_custom_model/using_a_custom_model/test_data/embeddings/2026-08-26_16-47___mel-test_data ###


{'audio_dir': 'bacpipe/tests/test_data',
 'embed_dir': 'bacpipe_results/using_a_custom_model/using_a_custom_model/test_data/embeddings/2026-08-26_16-47___mel-test_data',
 'embedding_size': 36096,
 'files': {'audio_files': ['audio/FewShot/CHE_01_20190101_163410.wav',
   'audio/FewShot/CHE_02_20190101_183410.wav',
   'audio/FewShot/CHE_03_20190201_163410.wav',
   'audio/FewShot/CHE_04_20190203_175410.wav',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031300.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031400.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031500.WAV'],
  'file_lengths (s)': [63.98977083333333,
   9.890729166666667,
   8.202479166666667,
   9.351708333333333,
   30.0,
   30.0,
   30.0],
  'nr_embeds_per_file': [22, 4, 3, 4, 10, 10, 10]},
 'model_name': 'mel',
 'nr_embeds_total': 63,
 'sample_rate (Hz)': 48000,
 'segment_length (samples)': 144000,
 'total_dataset_length (s)': 181.4346875}